# RL Policy Comparison Dashboard

This notebook is the main comparison cockpit for the sparse-action experiments. It compares:

- episodic survival on `test` and `train_eval` splits,
- action-0 use and non-idle action rate by agent,
- number of non-idle agents per environment step,
- heuristic override rate and blocked non-idle actions,
- gate diagnostics,
- sparse fixed penalty diagnostics,
- adaptive intervention-budget diagnostics,
- standalone full-test JSON summaries.

The notebook reads cached W&B full histories from `Topology_Task/outputs/wandb_cache`. If the cache is missing or stale, enable `DOWNLOAD_MISSING_FROM_WANDB` in the config cell.

In [ ]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    local = candidate / "comparison_dashboard.py"
    repo = candidate / "Topology_Task" / "configs" / "zz_print_metrics" / "comparison_dashboard.py"
    if local.exists():
        sys.path.insert(0, str(candidate))
        break
    if repo.exists():
        sys.path.insert(0, str(repo.parent))
        break

import comparison_dashboard as cd
cd = importlib.reload(cd)
print("comparison_dashboard:", cd.__file__)
print("TASK_DIR:", cd.TASK_DIR)

## Configuration

In [ ]:
CONFIG_FOLDERS = [
    "a0_hvg",
    "a0_sparse16",
    "a0_aib",
    # Historical / pre-a0 folders. Keep or remove depending on the comparison you want.
    "heuristic_vs_gate_s0_s1_s2",
    "phase4_sparse_control_16",
    "adaptive_intervention_budget_7",
    "adaptive_intervention_budget_mechanism_15",
]

DOWNLOAD_MISSING_FROM_WANDB = False
REFRESH_SCAN_HISTORY_FALLBACKS = False
FORCE_REFRESH_CACHE = False

STEP_MAX_M = None          # Example: 8.0 to compare only the first 8M steps.
SMOOTH_WINDOW = 5          # Rolling smoothing for curves; set to 1 for raw.
FINAL_LAST_N = 5           # Final summary = mean of last N logged points per run.
EVAL_SPLIT = "test"        # Usually "test". Can be "train_eval" for train split eval logs.
ACTION_SOURCE = "eval"     # "eval" uses test/explain action metrics; "train" uses rollout action metrics.
SHOW_FIGURES = True
SAVE_FIGURES = True

print("Config folders:", CONFIG_FOLDERS)

## Optional W&B Cache Refresh

Run this cell only when the cache does not contain the runs you want. It uses the existing `wandb_metrics.py` downloader.

In [ ]:
if DOWNLOAD_MISSING_FROM_WANDB:
    import wandb_metrics as wm
    wm = importlib.reload(wm)
    wm.configure_run_filter_from_config_folder(CONFIG_FOLDERS)
    wm.USE_LOCAL_CACHE_ONLY = False
    wm.REFRESH_SCAN_HISTORY_FALLBACKS = bool(REFRESH_SCAN_HISTORY_FALLBACKS)
    wm.FORCE_REFRESH = bool(FORCE_REFRESH_CACHE)
    wm.RUN_STATES = None
    wm.MAX_RUNS = None
    refreshed = wm.load_wandb_data()
    print("Downloaded/loaded W&B runs:", len(refreshed.runs_df))
else:
    print("Skipping W&B download; using local cache only.")

## Load Cached Histories And Check Coverage

In [ ]:
coverage = cd.cache_coverage(CONFIG_FOLDERS)
print(f"Expected configs: {len(coverage)}")
print(f"Cached configs: {int(coverage['cached'].sum())} / {len(coverage)}")

display(
    coverage.groupby(["experiment", "condition"], dropna=False)
    .agg(
        expected=("run_name", "count"),
        cached=("cached", "sum"),
        seeds=("seed", lambda s: sorted(s.dropna().astype(int).unique().tolist())),
    )
    .reset_index()
    .sort_values(["experiment", "condition"])
)

missing = coverage[~coverage["cached"]]
if not missing.empty:
    print("Missing cached histories:")
    display(missing[["experiment", "run_name", "condition", "seed", "config_path"]])

selected_runs = cd.select_cached_runs(CONFIG_FOLDERS)
history = cd.load_cached_histories(selected_runs, step_max_m=STEP_MAX_M, verbose=True)
print("history shape:", history.shape)

availability = cd.metric_availability(history)
display(availability)

## Episodic Survival Curves And Final Survival

Use this to compare baseline vs heuristic vs gate vs sparse/AIB directly.

In [ ]:
survival_tables = {}
for split in ["test", "train_eval"]:
    surv = cd.survival_curve(history, split=split, smooth=SMOOTH_WINDOW)
    survival_tables[split] = surv
    print(f"{split}: {len(surv)} logged survival rows")
    cd.plot_timeseries(
        surv,
        title=f"{split} episodic survival over training",
        y_title="episodic survival",
        save_name=f"policy_dashboard_{split}_survival_curve",
        save=SAVE_FIGURES,
        show=SHOW_FIGURES,
    )
    final = cd.per_run_final(surv, last_n=FINAL_LAST_N)
    display(cd.seed_aggregate(final)[["experiment", "condition", "mean", "std", "n_seeds", "seeds", "runs"]])
    cd.plot_final_bar(
        final,
        title=f"Final {split} survival, last {FINAL_LAST_N} logged points",
        y_title="episodic survival",
        save_name=f"policy_dashboard_final_{split}_survival",
        save=SAVE_FIGURES,
        show=SHOW_FIGURES,
    )

## Action 0 And Non-Idle Action Rate By Agent

`ACTION_SOURCE='eval'` answers what the deterministic policy actually does during evaluation, including heuristic overrides. `ACTION_SOURCE='train'` answers what the rollout exploration policy sampled during PPO training.

In [ ]:
for source, split in [(ACTION_SOURCE, EVAL_SPLIT), ("train", EVAL_SPLIT)]:
    print("source:", source)
    action0 = cd.agent_metric_curve(history, metric="action0", source=source, split=split, smooth=SMOOTH_WINDOW)
    nonidle = cd.agent_metric_curve(history, metric="nonidle", source=source, split=split, smooth=SMOOTH_WINDOW)
    cd.plot_timeseries(
        action0,
        title=f"Action-0 fraction by agent ({source})",
        y_title="fraction action 0",
        facet_col="agent",
        extra_group_cols=["agent"],
        save_name=f"policy_dashboard_action0_{source}_{split}",
        save=SAVE_FIGURES,
        show=SHOW_FIGURES,
        height=900,
    )
    final_action0 = cd.per_run_final(action0, last_n=FINAL_LAST_N, group_cols=("run_id", "agent"))
    display(cd.seed_aggregate(final_action0, extra_group_cols=["agent"])[["condition", "agent", "mean", "std", "n_seeds", "seeds"]])
    cd.plot_final_bar(
        final_action0,
        title=f"Final action-0 fraction by agent ({source})",
        y_title="fraction action 0",
        extra_group_cols=["agent"],
        color_col="agent",
        save_name=f"policy_dashboard_final_action0_{source}_{split}",
        save=SAVE_FIGURES,
        show=SHOW_FIGURES,
    )
    cd.plot_timeseries(
        nonidle,
        title=f"Non-idle action fraction by agent ({source})",
        y_title="non-idle fraction",
        facet_col="agent",
        extra_group_cols=["agent"],
        save_name=f"policy_dashboard_nonidle_{source}_{split}",
        save=SAVE_FIGURES,
        show=SHOW_FIGURES,
        height=900,
    )

## Number Of Non-Idle Agents Per Environment Step

This is the plot that tells whether actions are mostly single-agent interventions or simultaneous multi-agent interventions.

In [ ]:
joint = cd.joint_nonidle_curve(history, smooth=SMOOTH_WINDOW)
count_rows = joint[joint["non_idle_agents"].isin([0, 1, 2, 3])].copy()
coord_rows = joint[joint["metric"].isin(["train/frac_any_non_idle", "train/frac_multi_agent_non_idle", "train/non_idle_agents_mean"])].copy()

cd.plot_timeseries(
    count_rows,
    title="Training rollout distribution: number of non-idle agents per env step",
    y_title="fraction of env steps",
    facet_col="non_idle_agents",
    extra_group_cols=["non_idle_agents"],
    save_name="policy_dashboard_non_idle_agent_count_distribution",
    save=SAVE_FIGURES,
    show=SHOW_FIGURES,
    height=950,
)
final_counts = cd.per_run_final(count_rows, last_n=FINAL_LAST_N, group_cols=("run_id", "non_idle_agents"))
display(cd.seed_aggregate(final_counts, extra_group_cols=["non_idle_agents"])[["condition", "non_idle_agents", "mean", "std", "n_seeds", "seeds"]])
cd.plot_final_bar(
    final_counts,
    title=f"Final non-idle-agent count distribution, last {FINAL_LAST_N} logged points",
    y_title="fraction of env steps",
    extra_group_cols=["non_idle_agents"],
    color_col="non_idle_agents",
    save_name="policy_dashboard_final_non_idle_agent_count_distribution",
    save=SAVE_FIGURES,
    show=SHOW_FIGURES,
)

cd.plot_timeseries(
    coord_rows,
    title="Training rollout coordination metrics",
    y_title="value",
    facet_col="metric",
    extra_group_cols=["metric"],
    save_name="policy_dashboard_coordination_metrics",
    save=SAVE_FIGURES,
    show=SHOW_FIGURES,
    height=900,
)

## Heuristic Override Metrics

These metrics exist for heuristic-evaluation runs. They distinguish:

- `policy_nonidle`: the actor wanted a non-idle action before the heuristic,
- `force_noop`: the heuristic forced no-op,
- `blocked_nonidle`: the actor wanted non-idle and the heuristic blocked it.

In [ ]:
heur = cd.heuristic_curve(history, split=EVAL_SPLIT, smooth=SMOOTH_WINDOW)
print("heuristic rows:", len(heur))
if not heur.empty:
    final_heur = cd.per_run_final(heur, last_n=FINAL_LAST_N, group_cols=("run_id", "metric"))
    display(cd.seed_aggregate(final_heur, extra_group_cols=["metric"])[["condition", "metric", "mean", "std", "n_seeds", "seeds"]])

    for metric_filter, title in [
        (["force_noop_any_agent", "force_noop_all_agents"], "Heuristic force-noop rate: any/all agents"),
        ([f"force_noop_{a}" for a in cd.AGENTS], "Heuristic force-noop rate by agent"),
        ([f"policy_nonidle_{a}" for a in cd.AGENTS], "Pre-heuristic policy non-idle rate by agent"),
        ([f"blocked_nonidle_{a}" for a in cd.AGENTS], "Non-idle actions blocked by heuristic"),
    ]:
        data = heur[heur["metric"].isin(metric_filter)].copy()
        cd.plot_timeseries(
            data,
            title=title,
            y_title="fraction",
            facet_col="metric",
            extra_group_cols=["metric"],
            save_name="policy_dashboard_" + cd.safe_name(title),
            save=SAVE_FIGURES,
            show=SHOW_FIGURES,
            height=900,
        )

## Intervention Gate Diagnostics

In [ ]:
for metric, title in [
    ("gate_prob_intervene", "Gate predicted P(intervene)"),
    ("gate_actual_intervene", "Gate actual intervention fraction"),
    ("gate_entropy", "Gate entropy"),
]:
    gate_df = cd.agent_metric_curve(history, metric=metric, source="train", split=EVAL_SPLIT, smooth=SMOOTH_WINDOW)
    cd.plot_timeseries(
        gate_df,
        title=title,
        y_title="value",
        facet_col="agent",
        extra_group_cols=["agent"],
        save_name="policy_dashboard_" + cd.safe_name(title),
        save=SAVE_FIGURES,
        show=SHOW_FIGURES,
        height=900,
    )
    final_gate = cd.per_run_final(gate_df, last_n=FINAL_LAST_N, group_cols=("run_id", "agent"))
    if not final_gate.empty:
        display(cd.seed_aggregate(final_gate, extra_group_cols=["agent"])[["condition", "agent", "mean", "std", "n_seeds", "seeds"]])

## Fixed Sparse Penalty And Adaptive Budget Diagnostics

In [ ]:
sparse = cd.sparse_penalty_curve(history, smooth=SMOOTH_WINDOW)
if not sparse.empty:
    for metric in ["penalty_mean", "rate_when_safe", "rate_when_hazard", "safe_state_frac"]:
        data = sparse[sparse["metric"].eq(metric)].copy()
        cd.plot_timeseries(
            data,
            title=f"Sparse fixed-penalty diagnostic: {metric}",
            y_title=metric,
            facet_col="agent" if data["agent"].nunique() > 1 else None,
            extra_group_cols=["agent"] if data["agent"].nunique() > 1 else [],
            save_name=f"policy_dashboard_sparse_{metric}",
            save=SAVE_FIGURES,
            show=SHOW_FIGURES,
            height=850,
        )
else:
    print("No sparse fixed-penalty metrics found.")

aib = cd.aib_curve(history, smooth=SMOOTH_WINDOW)
if not aib.empty:
    for metric in ["lambda_mean", "cost_mean", "cost_violation_mean", "penalty_mean", "lambda", "cost", "rate_when_budget_costly", "rate_when_budget_free"]:
        data = aib[aib["metric"].eq(metric)].copy()
        cd.plot_timeseries(
            data,
            title=f"AIB diagnostic: {metric}",
            y_title=metric,
            facet_col="agent" if data["agent"].nunique() > 1 else None,
            extra_group_cols=["agent"] if data["agent"].nunique() > 1 else [],
            save_name=f"policy_dashboard_aib_{metric}",
            save=SAVE_FIGURES,
            show=SHOW_FIGURES,
            height=850,
        )
else:
    print("No AIB metrics found.")

## Illegal Action Rate

In [ ]:
illegal = cd.agent_metric_curve(history, metric="illegal", source="train", split=EVAL_SPLIT, smooth=SMOOTH_WINDOW)
cd.plot_timeseries(
    illegal,
    title="Training illegal action rate by agent",
    y_title="illegal action rate",
    facet_col="agent",
    extra_group_cols=["agent"],
    save_name="policy_dashboard_illegal_action_rate",
    save=SAVE_FIGURES,
    show=SHOW_FIGURES,
    height=900,
)
final_illegal = cd.per_run_final(illegal, last_n=FINAL_LAST_N, group_cols=("run_id", "agent"))
if not final_illegal.empty:
    display(cd.seed_aggregate(final_illegal, extra_group_cols=["agent"])[["condition", "agent", "mean", "std", "n_seeds", "seeds"]])

## Action Sparsity vs Survival Tradeoff

In [ ]:
tradeoff_per_run, tradeoff_agg = cd.action_survival_tradeoff(
    history,
    action_source=ACTION_SOURCE,
    split=EVAL_SPLIT,
    last_n=FINAL_LAST_N,
)
display(tradeoff_agg[["condition", "survival_mean", "survival_std", "action0_mean", "action0_std", "n_seeds", "seeds"]] if not tradeoff_agg.empty else tradeoff_agg)
cd.plot_action_survival_tradeoff(
    history,
    action_source=ACTION_SOURCE,
    split=EVAL_SPLIT,
    last_n=FINAL_LAST_N,
    save_name="policy_dashboard_action0_survival_tradeoff",
    save=SAVE_FIGURES,
    show=SHOW_FIGURES,
)

## Standalone Full-Test Eval JSON Summaries

This reads outputs created by `Topology_Task/full_test_eval/evaluate_checkpoint.py`. It is the place to compare the full chronic split, not only the short training eval.

In [ ]:
full_test = cd.load_full_test_results()
display(full_test.sort_values("survival_percent", ascending=False) if not full_test.empty else full_test)
cd.plot_full_test_results(full_test, show=SHOW_FIGURES, save=SAVE_FIGURES)

## Export Summary Tables

In [ ]:
out_dir = cd.FIG_DIR
out_dir.mkdir(parents=True, exist_ok=True)

if not tradeoff_agg.empty:
    tradeoff_agg.to_csv(out_dir / "policy_dashboard_tradeoff_summary.csv", index=False)
if not availability.empty:
    availability.to_csv(out_dir / "policy_dashboard_metric_availability.csv", index=False)
if not full_test.empty:
    full_test.to_csv(out_dir / "policy_dashboard_full_test_results.csv", index=False)

print("Wrote CSV summaries to", out_dir)